# **Pipeline for $k_{app}$** calculation

This notebook exemplifies the pipeline for the calculation of the apparent *in vivo* turnover number $k_{app}$ of *E. coli*. 

In [1]:
# automatic module reloading
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cobra
cobra.Configuration().solver = "gurobi" 

# Add paths
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../scripts'))

# Add directories
data_dir = "../data"

## 1. Create enzymes dataframe

In [2]:
# Module imports
from scripts.enzyme_classifier import create_gpr_dataframe, analyze_model_gprs

from cobra.io import read_sbml_model
import cobra 

# Load the model
model_path = os.path.join(data_dir, "raw", "GEMs", "iML1515_GEM.xml")
model = read_sbml_model(model_path)

# Create a dataframe of the GPR rules
df_enzymes = create_gpr_dataframe(model)

# Print stats of the model
stats = analyze_model_gprs(model)
print(f"\nModel Stats:")
print(f"Total reactions: {stats['total_reactions']}")
print(f"Reactions with GPR: {stats['reactions_with_gpr']}")
print(f"Total genes: {stats['total_genes']}")
print(f"GPR cases: {stats['gpr_complexity']}")
df_enzymes.head()

Set parameter Username
Set parameter LicenseID to value 2724784
Academic license - for non-commercial use only - expires 2026-10-18

Model Stats:
Total reactions: 2712
Reactions with GPR: 2266
Total genes: 1516
GPR cases: {'simple': 1302, 'or_only': 651, 'and_only': 221, 'complex': 92}


,gene,type,rxn,subsystem,subunit,GPR,enzyme_ID,gpr_class
0,b0870,isoenzyme,ALATA_D2,Cofactor and Prosthetic Group Biosynthesis,-,b2551 or b0870,b0870_i_ALATA_D2,or_only
1,b2551,isoenzyme,ALATA_D2,Cofactor and Prosthetic Group Biosynthesis,-,b2551 or b0870,b2551_i_ALATA_D2,or_only
2,b3368,homomeric,SHCHD2,Cofactor and Prosthetic Group Biosynthesis,-,b3368,b3368_h_SHCHD2,simple
3,b2436,homomeric,CPPPGO,Cofactor and Prosthetic Group Biosynthesis,-,b2436,b2436_h_CPPPGO,simple
4,b3500,homomeric,GTHOr,Cofactor and Prosthetic Group Biosynthesis,-,b3500,b3500_h_GTHOr,simple


## 2. Get fluxomics from pFBA simulations across multiple conditions

In [3]:
from scripts.kapp_builder import create_fluxomics_dataframe

fluxomics_df = create_fluxomics_dataframe(flux_method='pFBA', GEM=model, 
                                         carbon_uptake=[2, 6, 10], 
                                         oxygen_uptake=[15, 17.5, 20])


Processing condition 1: Carbon=2, Oxygen=15
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmpu2wqwfpv.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 1 completed successfully
Processing condition 2: Carbon=2, Oxygen=17.5
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmpg_95o3ju.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 2 completed successfully
Processing condition 3: Carbon=2, Oxygen=20
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmpodt90crl.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 3 completed successfully
Processing condition 4: Carbon=6, Oxygen=15
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmp3ctpjqir.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 4 completed successfully
Processing condition 5

In [4]:
flux_summary = fluxomics_df.set_index("rxn_id").agg(["mean", "std", "min", "max"], axis=1)

In [5]:
fluxomics_df

,rxn_id,flux_cond1,flux_cond2,flux_cond3,flux_cond4,flux_cond5,flux_cond6,flux_cond7,flux_cond8,flux_cond9
0,ALATA_D2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,SHCHD2,0.000035,0.000035,0.000035,0.000115,0.000115,0.000115,0.000154,0.000169,0.000183
2,CPPPGO,0.000035,0.000035,0.000035,0.000115,0.000115,0.000115,0.000154,0.000169,0.000183
3,GTHOr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,DHORD5,0.000000,0.000000,0.000000,0.000839,0.000839,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
2707,SUCCt1pp,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2708,QUINDH,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2709,LCARSyi,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2710,BIOMASS_Ec_iML1515_core_75p37M,0.154755,0.154755,0.154755,0.515876,0.515876,0.515876,0.692321,0.757060,0.821798


##  Get columns for FVA solutions of 'max_flux' and 'min_flux' across multiple conditions


In [6]:
from scripts.kapp_builder import create_FVA_dataframe

# Load the model again
model_path = os.path.join(data_dir, "raw", "GEMs", "iML1515_GEM.xml")

# Create FVA dataframe

fva_df = create_FVA_dataframe(
    GEM_path=model_path,
    carbon_uptake=[2, 6, 10],
    oxygen_uptake=[15, 17.5, 20],
    mu_fraction=0.9
)

Running FVA condition 1: Carbon=2, Oxygen=15
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmpe3f306xb.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 1 completed successfully.
Running FVA condition 2: Carbon=2, Oxygen=17.5
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmp5qmrmq3z.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 2 completed successfully.
Running FVA condition 3: Carbon=2, Oxygen=20
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmp5r4zp9zt.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 3 completed successfully.
Running FVA condition 4: Carbon=6, Oxygen=15
Read LP format model from file /var/folders/q9/9w_9rg4s5231v_bqmpbhy_bw0000gn/T/tmpu_qc329e.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Condition 4 completed successfully.
Running FVA co

In [7]:
print(fva_df.columns)


Index(['rxn_id', 'FVA_lower_cond1', 'FVA_lower_cond2', 'FVA_lower_cond3',
       'FVA_lower_cond4', 'FVA_lower_cond5', 'FVA_lower_cond6',
       'FVA_lower_cond7', 'FVA_lower_cond8', 'FVA_lower_cond9',
       'FVA_upper_cond1', 'FVA_upper_cond2', 'FVA_upper_cond3',
       'FVA_upper_cond4', 'FVA_upper_cond5', 'FVA_upper_cond6',
       'FVA_upper_cond7', 'FVA_upper_cond8', 'FVA_upper_cond9'],
      dtype='object')


In [8]:
# Merge fluxomics df with FVA bounds while filtering for violations

from scripts.kapp_builder import FVA_integration

# filtered_fluxomics_df, violations_df = FVA_integration(fluxomics_df, fva_df, filter=True)
filtered_fluxomics_df, violations_df = FVA_integration(fluxomics_df, fva_df, filter=True) 
fluxomics_df = filtered_fluxomics_df.copy() 
# violations_df = FVA_sanity_check(fluxomics_df, fva_df)

Detected 436 violations of FVA bounds.
Filtered out 100 reactions with violations.


In [9]:
violations_df.head(10)


,rxn_id,condition,flux,FVA_lower,FVA_upper,violation_type
0,EX_cobalt2_e,flux_cond1,-3.868884e-06,-3.868298e-06,-3.481996e-06,below_min
1,EX_cl_e,flux_cond1,-8.055017e-04,-8.055017e-04,-7.249515e-04,below_min
2,MEOHtrpp,flux_cond1,-3.095107e-07,-2.952479e-07,0.000000e+00,below_min
3,MEOHtex,flux_cond1,-3.095107e-07,-2.952479e-07,0.000000e+00,below_min
4,SHCHD2,flux_cond1,3.451045e-05,3.105940e-05,3.451045e-05,above_max
5,DM_amob_c,flux_cond1,3.095107e-07,0.000000e+00,2.801655e-07,above_max
6,DBTS,flux_cond1,3.095107e-07,0.000000e+00,3.010581e-07,above_max
7,EX_meoh_e,flux_cond1,3.095107e-07,0.000000e+00,2.932413e-07,above_max
8,MOBDabcpp,flux_cond1,1.083288e-06,9.749588e-07,1.081865e-06,above_max
9,MOBDtex,flux_cond1,1.083288e-06,9.749588e-07,1.081865e-06,above_max


In [ ]:
fluxomics_df.head()


,rxn_id,flux_cond1,flux_cond2,flux_cond3,flux_cond4,flux_cond5,flux_cond6,flux_cond7,flux_cond8,flux_cond9,...,FVA_lower_cond9,FVA_upper_cond1,FVA_upper_cond2,FVA_upper_cond3,FVA_upper_cond4,FVA_upper_cond5,FVA_upper_cond6,FVA_upper_cond7,FVA_upper_cond8,FVA_upper_cond9
0,ALATA_D2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,2.057272,2.057272,2.057272,5.310448,6.857906,6.857906,4.055155,4.751914,5.158263
2,CPPPGO,0.000035,0.000035,0.000035,0.000115,0.000115,0.000115,0.000154,0.000169,0.000183,...,0.0,0.023675,0.023675,0.023675,0.078920,0.078920,0.078920,0.166834,0.182435,0.198035
3,GTHOr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,2.277009,2.277009,2.277009,5.871464,7.328100,7.328100,5.161152,5.561929,5.962705
4,DHORD5,0.000000,0.000000,0.000000,0.000839,0.000839,0.000000,0.000000,0.000000,0.000000,...,0.0,0.324988,0.324988,0.324988,1.083347,1.083358,1.083358,1.517662,1.659577,1.801492
5,GLYCTO2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,5.418578,5.418578,5.418578,14.159080,18.285552,18.285552,11.585333,12.668667,13.752000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2706,FUMt1pp,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-1000.0,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
2707,SUCCt1pp,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-1000.0,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
2708,QUINDH,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2709,LCARSyi,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,12.534840,12.534840,12.534840,30.880897,38.259741,38.259741,18.245767,24.403229,30.235244


## 3. Merge fluxomics to enzyme, substrate, and sequence data

In [17]:
from scripts.kapp_builder import create_enzyme_info_dataframe

substrates_df = os.path.join(data_dir, "processed", "kinGEMs_iML1515", "iML1515_substrates.csv")
sequence_df = os.path.join(data_dir, "processed", "UniProt", "iML1515_E coli_83333_UniProt.csv")

enzymes_info_dfs = create_enzyme_info_dataframe(df_enzymes, fluxomics_df, substrates_df, sequence_df)


Processing 9 flux conditions...
Processing flux_cond1...
Rows before filtering: 4624
Completed flux_cond1: 921 rows after filtering
Processing flux_cond2...
Rows before filtering: 4624
Completed flux_cond2: 921 rows after filtering
Processing flux_cond3...
Rows before filtering: 4624
Completed flux_cond3: 921 rows after filtering
Processing flux_cond4...
Rows before filtering: 4624
Completed flux_cond4: 897 rows after filtering
Processing flux_cond5...
Rows before filtering: 4624
Completed flux_cond5: 897 rows after filtering
Processing flux_cond6...
Rows before filtering: 4624
Completed flux_cond6: 920 rows after filtering
Processing flux_cond7...
Rows before filtering: 4624
Completed flux_cond7: 903 rows after filtering
Processing flux_cond8...
Rows before filtering: 4624
Completed flux_cond8: 903 rows after filtering
Processing flux_cond9...
Rows before filtering: 4624
Completed flux_cond9: 903 rows after filtering
Created enzyme info dataframes for 9 conditions


`enzymes_info_dfs` is a dict storing all dataframes for each of the conditions created.

An example of how each  of these dfs look:

In [18]:
enzymes_info_dfs['flux_cond1'].head()

,gene,type,rxn,subsystem,subunit,GPR,enzyme_ID,gpr_class,rxn_id,flux_value,FVA_lower,FVA_upper,Reaction,SMILES,Direction,model_gene_id,uniprot_id,ec_number,sequence
6,b2436,homomeric,CPPPGO,Cofactor and Prosthetic Group Biosynthesis,-,b2436,b2436_h_CPPPGO,simple,CPPPGO,0.000035,0.00000,0.023675,CPPPGO,Cc1c2[nH]c(c1CCC(=O)[O-])Cc1[nH]c(c(CCC(=O)[O-...,forward,b2436,HEM6_ECOLI,1.3.3.3,MKPDAHQVKQFLLNLQDTICQQLTAVDGAEFVEDSWQREAGGGGRS...
34,b3916,homomeric,PFK_3,Pentose Phosphate Pathway,-,b3916,b3916_h_PFK_3,simple,PFK_3,0.013822,0.00000,5.942740,PFK_3,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,forward,b3916,PFKA_ECOLI,2.7.1.11,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...
35,b3916,homomeric,PFK_3,Pentose Phosphate Pathway,-,b3916,b3916_h_PFK_3,simple,PFK_3,0.013822,0.00000,5.942740,PFK_3,O=C(CO)[C@@H](O)[C@H](O)[C@H](O)[C@H](O)COP(=O...,forward,b3916,PFKA_ECOLI,2.7.1.11,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...
48,b4054,isoenzyme,LEUTAi,"Valine, Leucine, and Isoleucine Metabolism",-,b3770 or b4054,b4054_i_LEUTAi,or_only,LEUTAi,0.069722,0.06275,0.205846,LEUTAi,CC(C)CC(=O)C(=O)[O-],forward,b4054,TYRB_ECOLI,2.6.1.107; 2.6.1.57,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...
49,b4054,isoenzyme,LEUTAi,"Valine, Leucine, and Isoleucine Metabolism",-,b3770 or b4054,b4054_i_LEUTAi,or_only,LEUTAi,0.069722,0.06275,0.205846,LEUTAi,[NH3+]C(CCC(=O)[O-])C(=O)[O-],forward,b4054,TYRB_ECOLI,2.6.1.107; 2.6.1.57,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...


## 3. Map proteomics

In [19]:
from scripts.kapp_builder import process_enzyme_protein_mapping

paxdb_dir = os.path.join(data_dir, "raw", "PaxDb","511145_ecoli","511145-WHOLE_ORGANISM-integrated.txt")

enzyme_protein_info_dfs = process_enzyme_protein_mapping(enzymes_info_dfs, paxdb_dir, p_total=[0.32, 0.435, 0.55])


Loading PaxDB data from: ../data/raw/PaxDb/511145_ecoli/511145-WHOLE_ORGANISM-integrated.txt
PaxDB data loaded: 4094 rows
Processing 9 conditions × 3 p_total values = 27 combinations

Processing condition: flux_cond1
  Processing p_total=0.32 (1/27)
    Success: 921 rows, 916 with protein data
  Processing p_total=0.435 (2/27)
    Success: 921 rows, 916 with protein data
  Processing p_total=0.55 (3/27)
    Success: 921 rows, 916 with protein data

Processing condition: flux_cond2
  Processing p_total=0.32 (4/27)
    Success: 921 rows, 916 with protein data
  Processing p_total=0.435 (5/27)
    Success: 921 rows, 916 with protein data
  Processing p_total=0.55 (6/27)
    Success: 921 rows, 916 with protein data

Processing condition: flux_cond3
  Processing p_total=0.32 (7/27)
    Success: 921 rows, 916 with protein data
  Processing p_total=0.435 (8/27)
    Success: 921 rows, 916 with protein data
  Processing p_total=0.55 (9/27)
    Success: 921 rows, 916 with protein data

Processin

Example of how a df w proteomics looks:

In [20]:
# Stored with double index: condition, p_total
condition = 'flux_cond1'
p_total_val = 0.32
enzyme_protein_info_dfs[condition][p_total_val].head()

,gene,type,rxn,subsystem,subunit,GPR,enzyme_ID,gpr_class,rxn_id,flux_value,...,Direction,model_gene_id,uniprot_id,ec_number,sequence,protein_ppm,molecular_weight,protein_fraction,protein_mol_gdcw,protein_mmol_gdcw
0,b2436,homomeric,CPPPGO,Cofactor and Prosthetic Group Biosynthesis,-,b2436,b2436_h_CPPPGO,simple,CPPPGO,0.000035,...,forward,b2436,HEM6_ECOLI,1.3.3.3,MKPDAHQVKQFLLNLQDTICQQLTAVDGAEFVEDSWQREAGGGGRS...,13.8,34322.2093,0.000014,1.286630e-10,1.286630e-07
1,b3916,homomeric,PFK_3,Pentose Phosphate Pathway,-,b3916,b3916_h_PFK_3,simple,PFK_3,0.013822,...,forward,b3916,PFKA_ECOLI,2.7.1.11,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...,904.0,34841.6643,0.000904,8.302703e-09,8.302703e-06
2,b3916,homomeric,PFK_3,Pentose Phosphate Pathway,-,b3916,b3916_h_PFK_3,simple,PFK_3,0.013822,...,forward,b3916,PFKA_ECOLI,2.7.1.11,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...,904.0,34841.6643,0.000904,8.302703e-09,8.302703e-06
3,b4054,isoenzyme,LEUTAi,"Valine, Leucine, and Isoleucine Metabolism",-,b3770 or b4054,b4054_i_LEUTAi,or_only,LEUTAi,0.069722,...,forward,b4054,TYRB_ECOLI,2.6.1.107; 2.6.1.57,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,323.0,43537.2948,0.000323,2.374057e-09,2.374057e-06
4,b4054,isoenzyme,LEUTAi,"Valine, Leucine, and Isoleucine Metabolism",-,b3770 or b4054,b4054_i_LEUTAi,or_only,LEUTAi,0.069722,...,forward,b4054,TYRB_ECOLI,2.6.1.107; 2.6.1.57,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,323.0,43537.2948,0.000323,2.374057e-09,2.374057e-06


## 4. Calculate $k_{app}$ homomeric

In [21]:
from scripts.kapp_builder import calculate_kapp_homomeric

kapp_dfs = calculate_kapp_homomeric(enzyme_protein_info_dfs)


Processing condition: flux_cond1
  Processing p_total=0.32
    Rows before filtering homomeric: 921
    Rows after filtering homomeric: 341
    Rows before filtering duplicates: 341
    Rows after filtering duplicates: 320
    Calculated kcat_app for 313 enzymes
  Processing p_total=0.435
    Rows before filtering homomeric: 921
    Rows after filtering homomeric: 341
    Rows before filtering duplicates: 341
    Rows after filtering duplicates: 320
    Calculated kcat_app for 313 enzymes
  Processing p_total=0.55
    Rows before filtering homomeric: 921
    Rows after filtering homomeric: 341
    Rows before filtering duplicates: 341
    Rows after filtering duplicates: 320
    Calculated kcat_app for 313 enzymes

Processing condition: flux_cond2
  Processing p_total=0.32
    Rows before filtering homomeric: 921
    Rows after filtering homomeric: 341
    Rows before filtering duplicates: 341
    Rows after filtering duplicates: 320
    Calculated kcat_app for 313 enzymes
  Processin

In [22]:
# Stored with double index: condition, p_total
condition = 'flux_cond1'
p_total_val = 0.32
kapp_dfs[condition][p_total_val].head()

,gene,type,rxn,subsystem,subunit,GPR,enzyme_ID,gpr_class,rxn_id,flux_value,...,uniprot_id,ec_number,sequence,protein_ppm,molecular_weight,protein_fraction,protein_mol_gdcw,protein_mmol_gdcw,flux_value_per_sec,kcat_app
0,b2436,homomeric,CPPPGO,Cofactor and Prosthetic Group Biosynthesis,-,b2436,b2436_h_CPPPGO,simple,CPPPGO,0.000035,...,HEM6_ECOLI,1.3.3.3,MKPDAHQVKQFLLNLQDTICQQLTAVDGAEFVEDSWQREAGGGGRS...,13.8,34322.2093,0.000014,1.286630e-10,1.286630e-07,9.586235e-09,0.074507
1,b3916,homomeric,PFK_3,Pentose Phosphate Pathway,-,b3916,b3916_h_PFK_3,simple,PFK_3,0.013822,...,PFKA_ECOLI,2.7.1.11,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...,904.0,34841.6643,0.000904,8.302703e-09,8.302703e-06,3.839478e-06,0.462437
2,b3916,homomeric,PFK_3,Pentose Phosphate Pathway,-,b3916,b3916_h_PFK_3,simple,PFK_3,0.013822,...,PFKA_ECOLI,2.7.1.11,MIKKIGVLTSGGDAPGMNAAIRGVVRSALTEGLEVMGIYDGYLGLY...,904.0,34841.6643,0.000904,8.302703e-09,8.302703e-06,3.839478e-06,0.462437
11,b0243,homomeric,G5SD,Arginine and Proline Metabolism,-,b0243,b0243_h_G5SD,simple,G5SD,0.034209,...,PROA_ECOLI,1.2.1.41,MLEQMGIAAKQASYKLAQLSSREKNRVLEKIADELEAQSEIILNAN...,256.0,44629.5476,0.000256,1.835555e-09,1.835555e-06,9.502624e-06,5.176975
12,b0243,homomeric,G5SD,Arginine and Proline Metabolism,-,b0243,b0243_h_G5SD,simple,G5SD,0.034209,...,PROA_ECOLI,1.2.1.41,MLEQMGIAAKQASYKLAQLSSREKNRVLEKIADELEAQSEIILNAN...,256.0,44629.5476,0.000256,1.835555e-09,1.835555e-06,9.502624e-06,5.176975


## 5. Filter values above physical threshold

In [23]:
from scripts.kapp_builder import evaluate_kapp_homomeric

kapp_dfs = evaluate_kapp_homomeric(kapp_dfs)



Filtering kcat_app values outside range: 1e-05 to 1e+06 s⁻¹

Processing condition: flux_cond1
  Processing p_total=0.32
    Original rows: 320
    Filtered rows: 318
    Removed total: 2 (high: 2, low: 0)
    Removed high values range: 2.66e+10 to 2.66e+10 s⁻¹
  Processing p_total=0.435
    Original rows: 320
    Filtered rows: 318
    Removed total: 2 (high: 2, low: 0)
    Removed high values range: 1.96e+10 to 1.96e+10 s⁻¹
  Processing p_total=0.55
    Original rows: 320
    Filtered rows: 318
    Removed total: 2 (high: 2, low: 0)
    Removed high values range: 1.55e+10 to 1.55e+10 s⁻¹

Processing condition: flux_cond2
  Processing p_total=0.32
    Original rows: 320
    Filtered rows: 318
    Removed total: 2 (high: 2, low: 0)
    Removed high values range: 2.66e+10 to 2.66e+10 s⁻¹
  Processing p_total=0.435
    Original rows: 320
    Filtered rows: 318
    Removed total: 2 (high: 2, low: 0)
    Removed high values range: 1.96e+10 to 1.96e+10 s⁻¹
  Processing p_total=0.55
    Origi

## 6. Get $k_{max}$

In [24]:
from scripts.kapp_builder import get_kmax_homomeric

kmax_results = get_kmax_homomeric(kapp_dfs)

Starting kmax analysis across all conditions and p_total values...
  Added 311 valid entries from flux_cond1, p_total=0.32
  Added 311 valid entries from flux_cond1, p_total=0.435
  Added 311 valid entries from flux_cond1, p_total=0.55
  Added 311 valid entries from flux_cond2, p_total=0.32
  Added 311 valid entries from flux_cond2, p_total=0.435
  Added 311 valid entries from flux_cond2, p_total=0.55
  Added 311 valid entries from flux_cond3, p_total=0.32
  Added 311 valid entries from flux_cond3, p_total=0.435
  Added 311 valid entries from flux_cond3, p_total=0.55
  Added 312 valid entries from flux_cond4, p_total=0.32
  Added 312 valid entries from flux_cond4, p_total=0.435
  Added 312 valid entries from flux_cond4, p_total=0.55
  Added 312 valid entries from flux_cond5, p_total=0.32
  Added 312 valid entries from flux_cond5, p_total=0.435
  Added 312 valid entries from flux_cond5, p_total=0.55
  Added 311 valid entries from flux_cond6, p_total=0.32
  Added 311 valid entries from f

In [25]:
kmax_results.head()

,sequence,SMILES,kcat_app_max,condition_max,p_total_max,gene,rxn,flux_value,FVA_upper,FVA_lower,protein_mmol_gdcw,subsystem
0,MMITLRKLPLAVAVAAGVMSAQAMAVDFHGYARSGIGWTGSGGEQQ...,O=C[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO,4332.607908,flux_cond7,0.32,b4036,GLCtex_copy1,10.000000,1000.00000,0.00000,6.411330e-07,"Transport, Outer Membrane"
1,MDFSLTEEQELLLASIRELITTNFPEEYFRTCDQNGTYPREFMRAL...,C/C=C/C(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)...,3857.579892,flux_cond9,0.32,b1695,ACOAD1fr,0.292461,1000.00000,0.00000,2.105965e-08,Membrane Lipid Metabolism
2,MDFSLTEEQELLLASIRELITTNFPEEYFRTCDQNGTYPREFMRAL...,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,3857.579892,flux_cond9,0.32,b1695,ACOAD1fr,0.292461,1000.00000,0.00000,2.105965e-08,Membrane Lipid Metabolism
3,MPNITWCDLPEDVSLWPGLPLSLSGDEVMPLDYHAGRSGWLLYGRG...,[NH3+][C@@H](COP(=O)([O-])[O-])C(=O)[O-],1997.321299,flux_cond9,0.32,b4388,PSP_L,1.411713,8.45876,0.00000,1.963342e-07,Glycine and Serine Metabolism
4,MLKYRLISAFVLIPVVIAALFLLPPVGFAIVTLVVCMLAAWEWGQL...,CCCCCCC=CCCCCCCCC(=O)OCC(COP(=O)([O-])O)OC(=O)...,646.603533,flux_cond9,0.32,b0175,DASYN161,0.061811,5.21263,0.05563,2.655358e-08,Glycerophospholipid Metabolism


In [26]:
#kmax_results.to_csv(os.path.join(data_dir, "final", "kmax", "iml1515_homomeric_kmax.csv"), index=False)
kmax_results.to_csv(os.path.join(data_dir, "final", "kmax", "iml1515_homomeric_kmax_pFBA.csv"), index=False)

# Variability of η

In [27]:
from scripts.kapp_builder import get_eta

kapp_with_eta, kmax_with_variance = get_eta(kapp_dfs, kmax_results)

Calculating eta (kapp/kmax) for all conditions...

Processing condition: flux_cond1
  Processing p_total=0.32
    Calculated eta for 289 enzyme-substrate pairs
  Processing p_total=0.435
    Calculated eta for 289 enzyme-substrate pairs
  Processing p_total=0.55
    Calculated eta for 289 enzyme-substrate pairs

Processing condition: flux_cond2
  Processing p_total=0.32
    Calculated eta for 289 enzyme-substrate pairs
  Processing p_total=0.435
    Calculated eta for 289 enzyme-substrate pairs
  Processing p_total=0.55
    Calculated eta for 289 enzyme-substrate pairs

Processing condition: flux_cond3
  Processing p_total=0.32
    Calculated eta for 289 enzyme-substrate pairs
  Processing p_total=0.435
    Calculated eta for 289 enzyme-substrate pairs
  Processing p_total=0.55
    Calculated eta for 289 enzyme-substrate pairs

Processing condition: flux_cond4
  Processing p_total=0.32
    Calculated eta for 290 enzyme-substrate pairs
  Processing p_total=0.435
    Calculated eta for 2

In [28]:
# Checking eta for 2 dfs
df_cond1_p032 = kapp_with_eta['flux_cond1'][0.32]
df_cond2_p032 = kapp_with_eta['flux_cond2'][0.32]

In [29]:
from scripts.kcat_utils import plot_eta_variability

plot_eta_variability(kmax_with_variance)

ModuleNotFoundError: No module named 'rdkit'

In [ ]:
kmax_with_variance.to_csv(os.path.join(data_dir, "final", "kcat_app", "iml1515_homomeric_kmax_pFBA_variability.csv"), index=False)

# Comparison against in vitro

In [ ]:
from scripts.kcat_utils import load_kcat_dataset_ecoli

kmax_dir = os.path.join(data_dir, "results", "kmax", "iml1515_homomeric_kmax_pFBA.csv")
CPIPred_dir = os.path.join(data_dir, "raw", "CPIPred", "CPI_kcat_scrn.csv")
CatPred_dir = os.path.join(data_dir, "raw", "CatPred", "kcat-random_trainvaltest.csv")
EnzyExtract_dir = os.path.join(data_dir, "raw", "EnzyExtract", "EnzyExtractDB_176463.parquet")

df_kmax = pd.read_csv(kmax_dir)

df_cpi, df_catpred, df_enzyextract = load_kcat_dataset_ecoli(CPIPred_dir, CatPred_dir, EnzyExtract_dir)

In [ ]:
# Plot
from scripts.kcat_utils import compare_kcat_distribution

compare_kcat_distribution(df_kmax, "kcat_app_max", df_catpred, "kcat_CatPred", label1="kcat in vivo", label2="kcat in vitro")

In [ ]:
# Plot
from scripts.kcat_utils import compare_kcat_distribution

compare_kcat_distribution(df_kmax, "kcat_app_max", df_cpi, "kcat_CPIPred", label1="kmax_invivo", label2="kcat_CPIPred")

In [ ]:
# Plot
from scripts.kcat_utils import compare_kcat_distribution

compare_kcat_distribution(df_kmax, "kcat_app_max", df_enzyextract, "kcat_EnzyExtract", label1="kcat in vivo", label2="kcat EnzyExtract")

## Correct CatPred comparison
CatPred's SMILES include all of the involved substrates. I clean them to keep the main substrate(s) only, no cofactors, as independent entries. Then these and the in vivo substrates are canonicalized for a correct comparison.

In [ ]:
from scripts.kcat_utils import process_catpred_smiles

df_catpred_clean = df_catpred.copy()
df_catpred_clean.rename(columns={"SMILES": "reactant_smiles"}, inplace=True)
df_catpred_clean = process_catpred_smiles(df_catpred_clean)
df_catpred_clean.drop(columns={"reactant_smiles"}, inplace=True)


In [ ]:
compare_kcat_distribution(df_kmax, "kcat_app_max", df_catpred_clean, "kcat_CatPred", label1="kcat in vivo", label2="kcat in vitro")

In [ ]:
df_catpred_clean.to_csv(os.path.join(data_dir, "processed", "CatPred", "catpred_single_subs.csv"), index=False)

In [ ]:
# Merge with single substrate
df_kmax_catpred = pd.merge(kmax_results, df_catpred_clean, on=["sequence", "SMILES"], how='left')

# Only 3 matches after merging single substrates, canonicalizing is necessary

In [ ]:
from scripts.kcat_utils import canonicalize

df_catpred_clean['SMILES_canon'] = df_catpred_clean['SMILES'].apply(canonicalize)
kmax_results['SMILES_canon'] = kmax_results['SMILES'].apply(canonicalize)

In [ ]:
# Merge with canon
df_kmax_catpred2 = pd.merge(kmax_results, df_catpred_clean, on=["sequence", "SMILES_canon"], how='left')

# Just 4 datapoints merged 

# EnzyExtract comparison

In [ ]:
# We'll use the complete dataset for analysis
# This means considering orthologous enzymes - same genes but found in different species

df_enzyextract_complete = pd.read_parquet(EnzyExtract_dir)

In [ ]:

from scripts.kcat_utils import canonicalize

df_enzyextract_complete['SMILES_canon'] = df_enzyextract_complete['smiles'].apply(canonicalize)
df_kmax['SMILES_canon'] = df_kmax['SMILES'].apply(canonicalize)

In [ ]:
df_merge_enzyextract_canon =  pd.merge(df_kmax, df_enzyextract_complete, on=["sequence", "SMILES_canon"], how='left')

In [ ]:
df_merge_enzyextract_canon = df_merge_enzyextract_canon.dropna(subset=['kcat_value'])

In [ ]:
import numpy as np

# log transform 
log_kcat_app = np.log10(df_merge_enzyextract_canon['kcat_app_max'])
log_kcat_val = np.log10(df_merge_enzyextract_canon['kcat_value'])

# drop NaNs or -inf from log transform
valid = log_kcat_app.notna() & log_kcat_val.notna() & np.isfinite(log_kcat_app) & np.isfinite(log_kcat_val)

# pearson correlation (r)
r = log_kcat_app[valid].corr(log_kcat_val[valid], method='pearson')
print("Pearson r (log values):", r)

# R2
r2 = r ** 2
print("R2 (log values):", r2)


# [PRELIMINARY $k_{app}$]
*Keeping just for records*

Using fluxomics data matched to the Crown et al. study (E. coli,  W3110, M9+Glu none)

## a. Merge fluxomics information


In [ ]:
fba_exp_df = pd.read_csv(os.path.join(data_dir, "results", "ECOMICS", "exp_vs_FBA.csv"))

df_fluxomics_enzymes = pd.merge(
    df_enzymes,
    fba_exp_df,
    left_on="rxn",
    right_on="rxn_id",
    how="left"
)

df_fluxomics_enzymes = df_fluxomics_enzymes.drop(columns=["rxn_id"])

## b. Merge substrate partners
Using dataframe of substrate partners from kinGEMs pipeline

In [ ]:
# Load
iML1515_substrates_dir = os.path.join(data_dir, "processed", "kinGEMs_iML1515", "iML1515_substrates.csv")
df_iML1515_substrates = pd.read_csv(iML1515_substrates_dir)

# Keep just Reaction, SMILES and direction column
df_iML1515_substrates = df_iML1515_substrates[['Reaction', 'SMILES', 'Direction']]

In [ ]:
# Merge using reaction ID
df_fluxomics_enzymes = pd.merge(
    df_fluxomics_enzymes,
    df_iML1515_substrates,
    left_on="rxn",
    right_on="Reaction",
    how="left"
)

df_fluxomics_enzymes = df_fluxomics_enzymes.drop(columns=["Reaction"])

In [ ]:
# Drop rows with wrong direction-flux
df_fluxomics_enzymes = df_fluxomics_enzymes[
    ((df_fluxomics_enzymes['Direction'] == 'forward') & (df_fluxomics_enzymes['FBA_flux'] >= 0)) |
    ((df_fluxomics_enzymes['Direction'] == 'reverse') & (df_fluxomics_enzymes['FBA_flux'] <= 0))
]

print(len(df_fluxomics_enzymes))

In [ ]:
# Drop rows with balancing species - these are not substrates
# where SMILES is '[H+]' or 'O'
df_fluxomics_enzymes = df_fluxomics_enzymes[
    ~df_fluxomics_enzymes['SMILES'].isin(['[H+]', 'O'])
]

print(len(df_fluxomics_enzymes))

In [ ]:
# even stricter cofactor filtering 

# Define common cofactors to filter out
cofactors = {
    'O',           # water
    'O=O',         # molecular oxygen  
    '[H]',         # hydrogen
    'OO',          # hydrogen peroxide
    '[O]',         # atomic oxygen
    '[OH-]',       # hydroxide
    '[H+]',        # proton
    'N',           # nitrogen (sometimes used)
    'P',           # phosphorus
    'S',           # sulfur
}

df_fluxomics_enzymes = df_fluxomics_enzymes[
    ~df_fluxomics_enzymes['SMILES'].isin(cofactors)
]

print(len(df_fluxomics_enzymes))

In [ ]:
# even stricter cofactor filtering 

simple_cofactors = {
    'C(=O)O',      # formic acid
    'CO',          # methanol
    'CCO',         # ethanol
    'CC(=O)O',     # acetic acid
    'C',           # methane
    'CC',          # ethane
    'CCC',         # propane
    'N',           # ammonia (as N)
    'NN',          # hydrazine
    'C=O',         # formaldehyde
    'CC=O',        # acetaldehyde
    'O=C=O',       # carbon dioxide
    '[NH3+]',      # ammonium
    '[Na+]',       # sodium
    '[Cl-]',       # chloride
    '[K+]',        # potassium
    '[Mg+2]',      # magnesium
    '[Ca+2]',      # calcium
}

df_fluxomics_enzymes = df_fluxomics_enzymes[
    ~df_fluxomics_enzymes['SMILES'].isin(simple_cofactors)
]

print(len(df_fluxomics_enzymes))

In [ ]:
# Remove rows with 0 FBA flux 
df_fluxomics_enzymes = df_fluxomics_enzymes[df_fluxomics_enzymes['FBA_flux'] != 0]

## c. Add sequences from UniProt

In [ ]:
# Read sequences recovered from UniProt
seq_path = os.path.join(data_dir, "processed", 'UniProt', 'iML1515_E coli_83333_UniProt.csv')
seq_df = pd.read_csv(seq_path)

# Create a dictionary of gene-sequence
seq_mapping = seq_df.set_index('model_gene_id')['sequence'].to_dict()

# Map to dataframe
df_fluxomics_enzymes['sequence'] = df_fluxomics_enzymes['gene'].map(seq_mapping)

## d. . Map proteomics from PaxDb

In [ ]:
from scripts.paxdb_mapper import map_paxdb_to_gene

# Load PaxDB file
ecoli_paxdb_dir = os.path.join(data_dir,"raw", "PaxDb","511145_ecoli","511145-WHOLE_ORGANISM-integrated.txt")

# Convert to dataframe
ecoli_paxdb = pd.read_csv(
    ecoli_paxdb_dir,
    sep="\t",
    comment="#",
    header=None,
    names=["gene_name", "string_external_id", "abundance"]  # set column names
)

# Map PaxDB abundances to enzymes by gene ID and calculate protein concentrations
# p_total is the total protein content in g/gDCW
df_fluxomics_enzymes = map_paxdb_to_gene(ecoli_paxdb, df_fluxomics_enzymes, p_total=0.55)

## e. Filter dataframe to keep only homomeric enzymes (i.e. 'simple' gpr class)
TO DO: Add isoenzymes and complexes

In [ ]:
df_fluxomics_homomeric = df_fluxomics_enzymes[df_fluxomics_enzymes['gpr_class'] == 'simple']

In [ ]:
print(f'Length of ALL enzymes df: {len(df_fluxomics_enzymes)}')
print(f'Length of homomeric enzymes df: {len(df_fluxomics_homomeric)}')

In [ ]:
print(f'Length of homomeric enzymes df before dropping duplicates: {len(df_fluxomics_homomeric)}')

df_fluxomics_homomeric.drop_duplicates(subset=["SMILES", "sequence"])

print(f'Length of homomeric enzymes df after dropping duplicates: {len(df_fluxomics_homomeric)}')


## f. Calculate $k_{app}$

In [ ]:
# Convert negative fluxes to positive
df_fluxomics_homomeric.loc[:, 'FBA_flux'] = df_fluxomics_homomeric['FBA_flux'].abs()

# COBRA fluxes are in mmol/gDW*h
df_fluxomics_homomeric.loc[:, 'FBA_flux'] = df_fluxomics_homomeric['FBA_flux'] / 3600 # mmol/gDW*s

In [ ]:
# Divide flux (mmol/gDW*s) by enzyme concentration (mmol/gDCW) = kcat (1/s)
df_fluxomics_homomeric.loc[:, 'kcat_app'] = df_fluxomics_homomeric['FBA_flux'] / df_fluxomics_homomeric['protein_mmol_gdcw']

In [ ]:
# Filter 'kcat_app' to remove outlier enzymes above the physical threshold of 1e6
print(f'Length before threshold filtering: {len(df_fluxomics_homomeric)}')
threshold = 1e6
df_fluxomics_homomeric = df_fluxomics_homomeric[df_fluxomics_homomeric['kcat_app'] <= threshold]
print(f'Length after threshold filtering: {len(df_fluxomics_homomeric)}')


In [ ]:
# Export to csv
df_fluxomics_homomeric.to_csv(os.path.join(data_dir, "final", "kcat_app", "iml1515_homomeric_kcat_app.csv"), index=False)

## Comparisons to $in$ $vitro$: 
### CPI-Pred and CatPred kcat dfs

In [ ]:
# Load
CPIPred_dir = os.path.join(data_dir, "raw", "CPIPred", "CPI_kcat_scrn.csv")
CPIPred_df = pd.read_csv(CPIPred_dir)

CatPred_dir = os.path.join(data_dir, "raw", "CatPred", "kcat-random_trainvaltest.csv")
CatPred_df = pd.read_csv(CatPred_dir)

In [ ]:
# Keep only E coli data - this cuts too many enzymes!
CPIPred_df = pd.read_csv(CPIPred_dir)
CatPred_df = CatPred_df[(CatPred_df['taxonomy_id'] == 562) | (CatPred_df['taxonomy_id'] == 83333)]

In [ ]:
# Clean for easy merge
CPIPred_df = CPIPred_df[["SEQ", "CMPD_SMILES", "kcat"]]
CPIPred_df = CPIPred_df[CPIPred_df['kcat'].notna()]
CPIPred_df.rename(columns={"SEQ": "sequence", "CMPD_SMILES": "SMILES", "kcat": "kcat_CPIPred"}, inplace=True)

CatPred_df = CatPred_df[["sequence", "reactant_smiles", "value"]]
CatPred_df = CatPred_df[CatPred_df['value'].notna()]
CatPred_df.rename(columns={'reactant_smiles': 'SMILES', "value": "kcat_CatPred"}, inplace=True)

df_kcat = pd.concat([CPIPred_df, CatPred_df])

In [ ]:
# Read
df_fluxomics_homomeric_dir = os.path.join(data_dir, "final", "kcat_app", "iml1515_homomeric_kcat_app.csv")
df_fluxomics_homomeric = pd.read_csv(df_fluxomics_homomeric_dir)

# Merge with seq-smiles 
df_fluxomics_homomeric = pd.merge(df_fluxomics_homomeric, df_kcat, on=["sequence", "SMILES"], how='left')

# Merge with seq only 
#df_fluxomics_homomeric = pd.merge(df_fluxomics_homomeric, df_kcat, on=["sequence"], how='left')

# Merge with smiles only
#df_fluxomics_homomeric = pd.merge(df_fluxomics_homomeric, df_kcat, on=["SMILES"], how='left')

In [ ]:
# Plot
from scripts.kcat_utils import compare_kcat_distribution

compare_kcat_distribution(df_fluxomics_homomeric, "kcat_app", CPIPred_df, "kcat_CPIPred", label1="kcat_invivo", label2="kcat_CPIPred")

### Interpretation of quantile-quantile plot

When comparing in vivo (var $A$) to in vitro (var $B$)

**Intercept $a$:**
- Vertical shift of the regression line.
- Captures the systematic location difference between the two distributions.
    - Here we are comparing all quantiles of the in vivo distribution against the in vitro distribution. This will give us a 'systematic' measure - a consistent shift
- $a=0$ - distributions are aligned, no shift
- $a<1$ - var $B$ is systematically lower than $A$
    - On raw scale, ratio ≈ $10^a$
- $a>1$ - var $B$ is systematically higher than $A$

**Slope $b$:**
- Tilt of the regression line.
- Reflects the relative spread/dispersion of the two distributions.
- $b=1$ - similar spread
- $b<1$ - var $B$ is more compressed
- $b>1$ - var $B$ is more spread

**MAD of residuals (Median Absolute Deviation):**

- Measure of goodness of fit between the observed quantile pairs and the fitted regression line.
- Smaller MAD: the shift and scaling (a and b) explain the differences consistently across quantiles.
- Larger MAD: deviations are uneven, often driven by heavy tails or outliers.


In [ ]:
# Plot
from scripts.kcat_utils import compare_kcat_distribution

compare_kcat_distribution(df_fluxomics_homomeric, "kcat_app", CatPred_df, "kcat_CatPred", label1="kcat_invivo", label2="kcat_CatPred")

## ML Predicted kcats: CPI Pred

In [ ]:
# Load iML1515 CPI Pred predictions
CPIPred_predictions_dir = os.path.join(data_dir, "processed", "kinGEMs_iML1515", "CPI_kcat_predictions.csv")
CPIPred_predictions_df = pd.read_csv(CPIPred_predictions_dir)

# Drop kcat column - it's a placeholder
CPIPred_predictions_df.drop(columns=['kcat'], inplace=True)

# Get kcat by averaging predictions across 5 folds
CPIPred_predictions_df["kcat_pred"] = CPIPred_predictions_df[
    ["pred_value_0", "pred_value_1", "pred_value_2", "pred_value_3", "pred_value_4"]
].mean(axis=1)

# Keep only kcat_pred column
CPIPred_predictions_df.rename(columns={"SEQ": "sequence", "CMPD_SMILES": "SMILES"}, inplace=True)
CPIPred_predictions_df = CPIPred_predictions_df[["sequence", "SMILES", "kcat_pred"]]

# There are multiple predictions for the same seq-SMILES pair, keep the largest (as done in kinGEMs)
CPIPred_predictions_df = (
    CPIPred_predictions_df
    .groupby(["sequence", "SMILES"], as_index=False)
    .max()
)

In [ ]:
# Merge 
df_fluxomics_homomeric = pd.merge(df_fluxomics_homomeric, CPIPred_predictions_df, on=["sequence", "SMILES"], how='left')

In [ ]:
# Plot
from scripts.kcat_utils import compare_kcat_distribution

compare_kcat_distribution(df_fluxomics_homomeric, "kcat_app", CPIPred_predictions_df, "kcat_pred", label1="kcat_app", label2="kcat_CPIPred_predicted")

## (kinGEMs tuned) ML Predicted kcats: CPI Pred

In [ ]:
# Load kinGEMs tuned iML1515 CPI Pred predictions
CPIPred_tuned_dir = os.path.join(data_dir, "processed", "kinGEMs_iML1515", "fluxes_kcat_tuned.csv")
CPIPred_tuned_df = pd.read_csv(CPIPred_tuned_dir)

# Keep only kcat_pred column
CPIPred_tuned_df.rename(columns={"SEQ": "sequence", "CMPD_SMILES": "SMILES"}, inplace=True)
CPIPred_tuned_df = CPIPred_tuned_df[["sequence", "SMILES", "kcat_mean"]]

# Keep the largest kcat_mean for each seq-SMILES pair
CPIPred_tuned_df = (
    CPIPred_tuned_df
    .groupby(["sequence", "SMILES"], as_index=False)
    .max()
)

In [ ]:
# Merge 
df_fluxomics_homomeric = pd.merge(df_fluxomics_homomeric, CPIPred_tuned_df, on=["sequence", "SMILES"], how='left')

In [ ]:
# Plot
from scripts.kcat_utils import compare_kcat_distribution

compare_kcat_distribution(df_fluxomics_homomeric, "kcat_app", CPIPred_tuned_df, "kcat_mean", label1="kcat_app", label2="kcat_CPIPred_tuned")